#**1. CLIP**
CLIP은 이미지와 텍스트를 같은 의미 공간으로 정렬하기 위해 대비 학습(contrastive learning)을 사용하는 대표적인 Pretrained Vision–Language 모델입니다. 이미지 인코더와 텍스트 인코더를 각각 학습시킨 뒤, 올바른 이미지–텍스트 쌍은 임베딩 공간에서 가깝게, 관련 없는 쌍은 멀어지도록 학습함으로써 두 모달리티 간 의미적 대응 관계를 형성합니다. 이 구조 덕분에 별도의 태스크별 학습 없이도 텍스트로 정의한 클래스 설명만으로 이미지를 분류하는 zero-shot 분류가 가능하며, 이미지–텍스트 검색(Text-to-Image, Image-to-Text Retrieval)에 특히 강점을 보입니다. 다만 생성 능력은 없고, 주로 “이해와 정렬”에 초점이 맞춰진 모델이라는 특징을 가집니다. [[논문]](https://arxiv.org/pdf/2103.00020)

#**2. CLIP이 등장한 배경**
기존 컴퓨터 비전 모델은 다음과 같은 한계를 가지고 있었습니다.

- 데이터셋마다 고정된 라벨 구조에 의존
- 새로운 클래스가 나오면 재학습 필요
- 현실 세계처럼 열린(open-world) 개념을 다루기 어려움

CLIP은 이러한 문제를 해결하기 위해, 웹에 존재하는 대규모 이미지–텍스트 쌍 데이터를 활용하여 “이미지는 언어로 설명될 수 있고, 언어는 이미지로 대응될 수 있다” 라는 가정 하에 학습된 모델입니다.

#**3. CLIP의 핵심 아이디어 (Contrastive Learning)**
CLIP은 대비 학습(Contrastive Learning)을 사용합니다.

- 올바른 이미지–텍스트 쌍 → 임베딩 거리 가깝게
- 관계없는 이미지–텍스트 쌍 → 임베딩 거리 멀게

#**4. 모델 구조 (Architecture)**
CLIP은 두 개의 독립된 인코더로 구성됩니다.

- Vision Encoder
    - CNN(ResNet) 또는 ViT
    - 이미지를 하나의 벡터로 변환
- Text Encoder
    - Transformer
    - 문장을 하나의 벡터로 변환

두 인코더의 출력은 같은 차원의 임베딩 공간에 존재하며, 코사인 유사도(cosine similarity)를 통해 비교됩니다.

#**5.  학습 방식 (Training Objective)**
- 이미지 임베딩 ↔ 텍스트 임베딩 간 유사도 행렬 계산
- Softmax 기반 대칭적 cross-entropy loss 사용
    - Image → Text 방향
    - Text → Image 방향

이로 인해 CLIP은 이미지 검색. 텍스트 검색을 동시에 잘하는 구조가 됩니다.

#**6. Zero-shot Learning이 가능한 이유**
CLIP의 가장 큰 특징은 Zero-shot Image Classification입니다. 방법은 매우 단순합니다.

1. 분류하고 싶은 클래스들을 문장으로 표현
    - 예: “a photo of a dog”, “a photo of a cat”
2. 이미지 임베딩과 각 문장 임베딩의 유사도 계산
3. 가장 유사한 문장을 클래스 예측으로 선택

👉 즉, 라벨 = 텍스트 프롬프트 이 개념이 이후 LLM 기반 멀티모달 모델의 핵심 사고방식이 됩니다.

<img src ='https://img1.daumcdn.net/thumb/R1280x0/?scode=mtistory2&fname=https%3A%2F%2Fblog.kakaocdn.net%2Fdna%2FboDbuT%2FdJMcahpLQc1%2FAAAAAAAAAAAAAAAAAAAAAHH9tv2DNrWQSEMiBZ71P7YNKsS9AQ1nb0_c7HJaJzGu%2Fimg.png%3Fcredential%3DyqXZFxpELC7KVnFOS48ylbz2pIh7yKj8%26expires%3D1772290799%26allow_ip%3D%26allow_referer%3D%26signature%3DNYuAboo1YWx7%252FS5dM2JjLnVYv24%253D'>

#**7. CLIP 모델로 코사인 유사도를 통해 히트맵으로 시각화**
> (행 = 이미지, 열 = 라벨 문장)인 10*10 행렬을 만들고, 각 칸에 점수를 써서 히트맵으로 저장

1. CLIP 모델 로드
2. ImageNetV2 데이터셋에서 테스트 split를 가져오고 10개 랜덤 샘플링
3. classnames.txt로부터 클래스 id와 클래스 이름 매핑
4. 샘플 이미지들을 PIL 이미지로 변환
5. 각 샘플의 정답 클래스 이름을 "a photo of a {class}" 형태의 텍스트 프롬프트로 변환
6. CLIP으로 이미지 임베딩 10개 추출, 텍스트 임베딩 10개 추출
7. 임베딩을 정규화해서 코사인 유사도를 계산
8. 유사도 행렬을 히트맵으로 그리고 셀에 숫자 표기, 왼쪽에 이미지 썸네일을 붙임
9. 이미지 저장, 화면에 출력

In [ ]:
import io
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
import seaborn as sns
from PIL import Image
from datasets import load_dataset
from transformers import CLIPProcessor, CLIPModel
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch32"
DATASET_NAME = "clip-benchmark/wds_imagenetv2"
SPLIT = "test"
SAMPLE_SIZE = 10
THUMB_SIZE = (80, 80)
OUT_PATH = "heatmap.png"

1. ImageNet
    - 1,000개의 클래스
    - 약 120만 장 학습 이미지
    - 컴퓨터 비전 분류의 표준 벤치마크
    - 모델이 너무 오래 ImageNet에 최적화되다보니 "진짜 일반화"인지, "ImageNet 특화"인지 구분이 어려워짐

2. ImageNetV2
    - 1,000개의 클래스
    - ImageNet에 사용되지 않은 새로운 이미지들로 다시 수집한 데이터셋
    - "모델이 진짜로 잘 맞추는거야? 아니면 ImageNet에 과적합된 거야?"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model.eval()

In [ ]:
# 데이터셋을 불러오고 test split에서 섞어서 10개만 고름
dataset = load_dataset(DATASET_NAME)
ds = dataset[SPLIT]
subset = ds.shuffle(seed = 2026).select(range(SAMPLE_SIZE))

In [ ]:
cls2label = [line.strip() for line in open("classnames.txt", "r", encoding = "utf-8").readlines()]
cls2label

In [ ]:
def to_pil(x):
    if isinstance(x, Image.Image):
        return x.convert("RGB")
        # subset에서 webp 칼럼을 꺼내 PIL 이미지 리스트를 만듦
        # images = [to_pil(x) for x in list(subset("webp"))]
    if isinstance(x, dict) and "bytes" in x and x["bytes"] is not None:
        return Image.open(io.BytesIO(x["bytes"])).convert("RGB")
    if isinstance(x, str):
        return Image.open(x).convert("RGB")
    raise TypeError(f"Unsupported image type: {type(x)}")

In [ ]:
label_names = [cls2label[int(c)] for c in list(subset['cls'])]
label_texts = [f"a photo of a {name}" for name in label_names]

images = [to_pil(x) for x in list(subset['webp'])]

In [ ]:
images

In [ ]:
label_texts

In [ ]:
with torch.no_grad():
    # processor가 이미지를 CLIP이 받는 파이토치 텐서 형태로 변환(batch, 3, H, W)
    inputs_image = processor(images = images, return_tensors = 'pt', padding = True).to(device)
    # vision_model이 비전 트랜스포머를 통과시켜 출력
    vision_out = model.vision_model(pixel_values = inputs_image['pixel_values'])
    # pooler_output = "대표 임베딩" 같은 역할을 하는 벡터(보통 CLS 토큰 기반)
    # visual_projection으로 CLIP 공통 임베딩 공간에 차원에 맞춰 projection 함
    image_features = model.visual_projection(vision_out.pooler_output)

    # 텍스트 임베딩
    # 텍스트도 토크나이징/패딩 후 모델 통과
    inputs_text = processor(text = label_texts, return_tensors = 'pt', padding = True).to(device)
    text_out = model.text_model(
        input_ids = inputs_text['input_ids'],
        attention_mask = inputs_text['attention_mask']
    )
    text_features = model.text_projection(text_out.pooler_output)

# 정규화 + 유사도 계산(코사인 유사도)
# 각 벡터를 L2 정규화하면 길이가 1이 됨
# 내적(@)이 코사인 유사도
# 결과가 (10, D) @ (D, 10) = (10, 10)
image_features = image_features / image_features.norm(dim = -1, keepdim = True)
text_features = text_features / text_features.norm(dim = -1, keepdim = True)
similarity_matrix = (image_features @ text_features.T).cpu().numpy()

In [ ]:
def create_thumbnail(img, size = (80, 80)):
    return img.resize(size)

In [ ]:
thumbnails = [create_thumbnail(img, THUMB_SIZE) for img in images]
thumbnails

In [ ]:
fig, ax = plt.subplots(figsize = (20, 12))
im = ax.imshow(similarity_matrix, aspect = 'auto')
plt.colorbar(im, ax = ax, fraction = 0.02, pad = 0.02)

ax.set_xticks(np.arange(len(label_texts)))
ax.set_xticklabels(label_texts, rotation = 45, ha = 'right')

ax.set_yticks(np.arange(len(images)))
ax.set_yticklabels([""] * len(images))

ax.set_title('CLIP Image vs. Label Similarity (Cosine Similarity)')
ax.set_xlabel('Label Text')
ax.set_ylabel('Image')

for i in range(similarity_matrix.shape[0]):
    for j in range(similarity_matrix.shape[1]):
        ax.text(j, i, f'{similarity_matrix[i, j]:.2f}', ha = 'center', va = 'center', fontsize = 8)

for i, img in enumerate(thumbnails):
    imagebox = OffsetImage(img, zoom = 1.0)
    # AnnotationBbox를 사용해서 좌표 (-0.6, i) 위치에 썸네일 이미지를 붙임
    # (-0.6, i)는 히트맵의 첫 열(0)보다 왼쪽에 배치하기 위해서
    ab = AnnotationBbox(imagebox, (-0.6, i), frameon = False, xycoords = 'data', boxcoords = 'data', pad = 0)
    ax.add_artist(ab)

ax.set_xlim(-1.2, similarity_matrix.shape[1] - 0.5)
ax.set_ylim(similarity_matrix.shape[1] - 0.5, - 0.5)

plt.tight_layout()
plt.savefig(OUT_PATH, dpi = 200)
plt.show()
print(f'saved: {OUT_PATH}')

* 각 행에서 정답 라벨 열의 값이 가장 높아야 함
* 즉, 대각선 성분이 높고 나머지가 낮으면 잘 맞음
* 대각선이 아닌 곳이 높으면 그 이미지를 CLIP이 다른 클래스로 더 강하게 매칭했다고 볼 수 있음